In [2]:
from pathlib import Path
from transformers import AutoTokenizer

In [3]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [6]:
print(f"Loaded tokenizer: {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

Loaded tokenizer: TokenizersBackend
Vocab size: 128,000


In [7]:
rules_path = Path("annotation_rules.md")
rules_text = rules_path.read_text(encoding="utf-8")

In [8]:
print(f"File: {rules_path.name}")
print(f"Characters: {len(rules_text):,}")
print(f"Words: {len(rules_text.split()):,}")
print(f"Lines: {len(rules_text.splitlines()):,}")

File: annotation_rules.md
Characters: 5,600
Words: 812
Lines: 112


In [12]:
tokens = tokenizer.encode(rules_text, add_special_tokens=True)
len(tokens)

1033

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

csv_path = Path("annotated_dataset.csv")
df = pd.read_csv(csv_path)

# Combine type and ticket into the exact input representation
tickets = ("Type: " + df["type"].fillna("") + "\n\nTicket: " + df["ticket"].fillna("")).tolist()

print(f"Loaded {len(tickets):,} tickets from {csv_path.name}")

/home/arnav-gupta/Projects/itops_agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 7,343 tickets from annotated_dataset.csv


In [3]:
deepseek_model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
deepseek_tok = AutoTokenizer.from_pretrained(deepseek_model_id, local_files_only=True)

In [4]:
print(f"DeepSeek-R1 Tokenizer: {deepseek_tok.__class__.__name__} (Vocab size: {deepseek_tok.vocab_size:,})")

DeepSeek-R1 Tokenizer: TokenizersBackend (Vocab size: 128,000)


In [7]:
# Laya Tokenizer (ModernBERT backbone)
candidate_paths = [
    Path("../../../../laya_work/laya_finetuned_itops_complexity/tokenizer"),
    Path("../../../laya_work/laya_finetuned_itops_complexity/tokenizer"),
    Path("laya_work/laya_finetuned_itops_complexity/tokenizer"),
    Path("/home/arnav-gupta/Projects/itops_agent/laya_work/laya_finetuned_itops_complexity/tokenizer")
]
laya_tokenizer_path = None
for p in candidate_paths:
    if p.resolve().exists():
        laya_tokenizer_path = str(p.resolve())
        break
if laya_tokenizer_path is None:
    raise FileNotFoundError("Could not locate Laya tokenizer directory!")
laya_tok = AutoTokenizer.from_pretrained(laya_tokenizer_path, local_files_only=True)

In [8]:
print(f"Laya Tokenizer: {laya_tok.__class__.__name__} (Vocab size: {laya_tok.vocab_size:,})")

Laya Tokenizer: TokenizersBackend (Vocab size: 50,280)


In [10]:
# Vectorized tokenization across all 7,343 rows (< 2 seconds)
print("Tokenizing entire dataset with DeepSeek-R1...")
deepseek_lengths = np.array([len(ids) for ids in deepseek_tok(tickets, add_special_tokens=False)["input_ids"]])

print("Tokenizing entire dataset with Laya...")
laya_lengths = np.array([len(ids) for ids in laya_tok(tickets, add_special_tokens=False)["input_ids"]])

print(f"Successfully computed token lengths for {len(deepseek_lengths):,} samples.")

Tokenizing entire dataset with DeepSeek-R1...
Tokenizing entire dataset with Laya...
Successfully computed token lengths for 7,343 samples.


In [12]:
metrics_data = [
    ("Total Samples Evaluated", f"{len(deepseek_lengths):,} tickets"),
    ("Total Corpus Tokens", f"{int(np.sum(deepseek_lengths)):,} tokens"),
    ("Mean Length (Average)", f"{np.mean(deepseek_lengths):.2f} tokens/ticket"),
    ("Standard Deviation (σ)", f"{np.std(deepseek_lengths):.2f}"),
    ("Median Length (p50)", f"{np.median(deepseek_lengths):.1f} tokens"),
    ("Interquartile Range (p25 - p75)", f"{np.percentile(deepseek_lengths, 25):.1f} – {np.percentile(deepseek_lengths, 75):.1f} tokens"),
    ("95th Percentile (p95)", f"{np.percentile(deepseek_lengths, 95):.1f} tokens"),
    ("99th Percentile (p99)", f"{np.percentile(deepseek_lengths, 99):.1f} tokens"),
    ("Minimum Ticket Length", f"{int(np.min(deepseek_lengths))} tokens"),
    ("Maximum Ticket Length", f"{int(np.max(deepseek_lengths))} tokens"),
    ("Model Context Window", "131,072 tokens (128k)"),
    ("Max Context Utilization", f"{np.max(deepseek_lengths) / 131072 * 100:.3f}% (100% Fit, Zero Truncation)")
]
ds_token_report = pd.DataFrame(metrics_data, columns=["Token Metric", "DeepSeek-R1 (8B) Value"])
display(ds_token_report)
print("\nMarkdown Table:")
print(ds_token_report.to_markdown(index=False))

,Token Metric,DeepSeek-R1 (8B) Value
0,Total Samples Evaluated,"7,343 tickets"
1,Total Corpus Tokens,"552,016 tokens"
2,Mean Length (Average),75.18 tokens/ticket
3,Standard Deviation (σ),30.79
4,Median Length (p50),77.0 tokens
5,Interquartile Range (p25 - p75),50.0 – 102.0 tokens
6,95th Percentile (p95),118.0 tokens
7,99th Percentile (p99),129.0 tokens
8,Minimum Ticket Length,10 tokens
9,Maximum Ticket Length,187 tokens



Markdown Table:
| Token Metric                    | DeepSeek-R1 (8B) Value             |
|:--------------------------------|:-----------------------------------|
| Total Samples Evaluated         | 7,343 tickets                      |
| Total Corpus Tokens             | 552,016 tokens                     |
| Mean Length (Average)           | 75.18 tokens/ticket                |
| Standard Deviation (σ)          | 30.79                              |
| Median Length (p50)             | 77.0 tokens                        |
| Interquartile Range (p25 - p75) | 50.0 – 102.0 tokens                |
| 95th Percentile (p95)           | 118.0 tokens                       |
| 99th Percentile (p99)           | 129.0 tokens                       |
| Minimum Ticket Length           | 10 tokens                          |
| Maximum Ticket Length           | 187 tokens                         |
| Model Context Window            | 131,072 tokens (128k)              |
| Max Context Utilization         

In [13]:
laya_metrics_data = [
    ("Total Samples Evaluated", f"{len(laya_lengths):,} tickets"),
    ("Total Corpus Tokens", f"{int(np.sum(laya_lengths)):,} tokens"),
    ("Mean Length (Average)", f"{np.mean(laya_lengths):.2f} tokens/ticket"),
    ("Standard Deviation (σ)", f"{np.std(laya_lengths):.2f}"),
    ("Median Length (p50)", f"{np.median(laya_lengths):.1f} tokens"),
    ("Interquartile Range (p25 - p75)", f"{np.percentile(laya_lengths, 25):.1f} – {np.percentile(laya_lengths, 75):.1f} tokens"),
    ("95th Percentile (p95)", f"{np.percentile(laya_lengths, 95):.1f} tokens"),
    ("99th Percentile (p99)", f"{np.percentile(laya_lengths, 99):.1f} tokens"),
    ("Minimum Ticket Length", f"{int(np.min(laya_lengths))} tokens"),
    ("Maximum Ticket Length", f"{int(np.max(laya_lengths))} tokens"),
    ("Model Context Limit (`max_len`)", "512 tokens"),
    ("Max Context Utilization", f"{np.max(laya_lengths) / 512 * 100:.2f}% (100% Fit, Zero Truncation)")
]
# Display Paperwork-Ready Table
laya_token_report = pd.DataFrame(laya_metrics_data, columns=["Token Metric", "Laya (ModernBERT) Value"])
display(laya_token_report)
print("\nMarkdown Table:")
print(laya_token_report.to_markdown(index=False))

,Token Metric,Laya (ModernBERT) Value
0,Total Samples Evaluated,"7,343 tickets"
1,Total Corpus Tokens,"595,328 tokens"
2,Mean Length (Average),81.07 tokens/ticket
3,Standard Deviation (σ),31.69
4,Median Length (p50),83.0 tokens
5,Interquartile Range (p25 - p75),55.0 – 108.0 tokens
6,95th Percentile (p95),126.0 tokens
7,99th Percentile (p99),138.0 tokens
8,Minimum Ticket Length,12 tokens
9,Maximum Ticket Length,200 tokens



Markdown Table:
| Token Metric                    | Laya (ModernBERT) Value            |
|:--------------------------------|:-----------------------------------|
| Total Samples Evaluated         | 7,343 tickets                      |
| Total Corpus Tokens             | 595,328 tokens                     |
| Mean Length (Average)           | 81.07 tokens/ticket                |
| Standard Deviation (σ)          | 31.69                              |
| Median Length (p50)             | 83.0 tokens                        |
| Interquartile Range (p25 - p75) | 55.0 – 108.0 tokens                |
| 95th Percentile (p95)           | 126.0 tokens                       |
| 99th Percentile (p99)           | 138.0 tokens                       |
| Minimum Ticket Length           | 12 tokens                          |
| Maximum Ticket Length           | 200 tokens                         |
| Model Context Limit (`max_len`) | 512 tokens                         |
| Max Context Utilization         